# 05-Watermarking du modèle : robustesse à l'int8, au pruning et au fine-tuning

In [1]:
import sys
sys.path.append("..")

import json
import torch
from torch.utils.data import Subset

from src.data import load_mnist
from src.model import CNN
from src.evaluate import evaluate
from src.watermark import TriggerSet, train, train_with_watermark, quantize_int8, prune_model, finetune

In [2]:
torch.manual_seed(0)
train_set, test = load_mnist()
trigger = TriggerSet()

clean = train(CNN(), train_set)
wm = train_with_watermark(CNN(), train_set, trigger)

100.0%
100.0%
100.0%
100.0%


In [3]:
# l'attaquant n'a qu'une partie des données
attack_data = Subset(train_set, range(10000))

models = {
    "clean": clean,
    "watermarked": wm,
    "wm + int8": quantize_int8(wm),
    "wm + pruning 50%": prune_model(wm, 0.5),
    "wm + fine-tuning": finetune(wm, attack_data),
}

results = {}
for name, m in models.items():
    results[name] = {"test": evaluate(m, test), "watermark": evaluate(m, trigger)}
    print(f"{name:18s} test {results[name]['test']:.4f}   watermark {results[name]['watermark']:.4f}")

with open("../results/watermarking.json", "w") as f:
    json.dump(results, f, indent=2)

clean              test 0.9817   watermark 0.1200
watermarked        test 0.9865   watermark 0.9800
wm + int8          test 0.9869   watermark 0.9800
wm + pruning 50%   test 0.9869   watermark 0.9300
wm + fine-tuning   test 0.9889   watermark 1.0000
